In [ ]:
#import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import pearsonr
import geopandas as gpd
from shapely.geometry import Point


In [ ]:
# use forward slashes (or raw strings) to avoid backslash escape issues on Windows paths
df_prec = pd.read_csv('../../ExtractedDatasets/ClimateFeatures/prec_2020_2024.csv')
df_tmax = pd.read_csv('../../ExtractedDatasets/ClimateFeatures/tmax_2020_2024.csv')
df_tmin = pd.read_csv('../../ExtractedDatasets/ClimateFeatures/tmin_2020_2024.csv')

In [ ]:
#Data shapes
print("Precipitation Data Shape:", df_prec.shape)
print("Max Temperature Data Shape:", df_tmax.shape)
print("Min Temperature Data Shape:", df_tmin.shape)

In [ ]:
#Data memory usage
print("Precipitation Data Memory Usage:", df_prec.memory_usage(deep=True).sum() / (1024 ** 2), "MB")
print("Max Temperature Data Memory Usage:", df_tmax.memory_usage(deep=True).sum() / (1024 ** 2), "MB")
print("Min Temperature Data Memory Usage:", df_tmin.memory_usage(deep=True).sum() / (1024 ** 2), "MB")

In [ ]:
# decribing datasets
print("Precipitation Data Description:")
print(df_prec.describe())
print("\nTmax Data Description:")
print(df_tmax.describe())
print("\nTmin Data Description:")
print(df_tmin.describe())

In [ ]:
# functionto calculate % of missing values
def missing_percentage(df):
    missing_count = df.isnull().sum().sum()
    total_count = df.size
    return (missing_count / total_count) * 100

In [ ]:
print("Precipitation Data Missing Percentage: {:.2f}%".format(missing_percentage(df_prec)))
print("Max Temperature Data Missing Percentage: {:.2f}%".format(missing_percentage(df_tmax)))
print("Min Temperature Data Missing Percentage: {:.2f}%".format(missing_percentage(df_tmin)))

In [ ]:
#display rows where there are missing values
df_missing_prec = df_prec[df_prec.isnull().any(axis=1)]

df_missing_tmax = df_tmax[df_tmax.isnull().any(axis=1)]

df_missing_tmin = df_tmin[df_tmin.isnull().any(axis=1)]

In [ ]:
#check what columns have missing values
print("Columns with missing values in Precipitation Data:")
print(df_missing_prec.columns[df_missing_prec.isnull().any()].tolist())
print("\nColumns with missing values in Max Temperature Data:")
print(df_missing_tmax.columns[df_missing_tmax.isnull().any()].tolist())
print("\nColumns with missing values in Min Temperature Data:")
print(df_missing_tmin.columns[df_missing_tmin.isnull().any()].tolist())

In [ ]:
# compare cell_id across the three missing-dataframes and show rows where they differ
cols = ['cell_id']
df_cmp = pd.concat([df_missing_prec[cols], df_missing_tmax[cols], df_missing_tmin[cols]], axis=1, keys=['prec','tmax','tmin'])
df_cmp.columns = ['prec_cell_id', 'tmax_cell_id', 'tmin_cell_id']

# rows where not all three cell_id values are equal
equal_mask = (df_cmp['prec_cell_id'] == df_cmp['tmax_cell_id']) & (df_cmp['prec_cell_id'] == df_cmp['tmin_cell_id'])
diff_idx = df_cmp.index[~equal_mask]

if len(diff_idx) == 0:
    print("All cell_id values are identical across df_missing_prec, df_missing_tmax and df_missing_tmin.")
else:
    print(f"{len(diff_idx)} rows have differing cell_id values. Showing full rows from each dataframe for those indices:")
    df_diff_full = pd.concat([
        df_missing_prec.loc[diff_idx].add_prefix('prec_'),
        df_missing_tmax.loc[diff_idx].add_prefix('tmax_'),
        df_missing_tmin.loc[diff_idx].add_prefix('tmin_')
    ], axis=1)
    df_diff_full

In [ ]:
# combine indices of rows that had missing values and drop them from the original dataframes
missing_idx = df_missing_prec.index.union(df_missing_tmax.index).union(df_missing_tmin.index)
print(f"Dropping {len(missing_idx)} rows with missing values from each dataframe.")

df_prec.drop(index=missing_idx, inplace=True)
df_tmax.drop(index=missing_idx, inplace=True)
df_tmin.drop(index=missing_idx, inplace=True)

# quick verification
print("Remaining missing in prec_mm:", df_prec['prec_mm'].isnull().sum())
print("Remaining missing in tmax_c:", df_tmax['tmax_c'].isnull().sum())
print("Remaining missing in tmin_c:", df_tmin['tmin_c'].isnull().sum())

In [ ]:
#visualization with histograms
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
sns.histplot(df_prec['prec_mm'], bins=50, kde=True, color='blue')
plt.title('Precipitation Distribution')
plt.xlabel('Precipitation (mm)')
plt.ylabel('Frequency')
plt.subplot(1, 3, 2)
sns.histplot(df_tmax['tmax_c'], bins=50, kde=True, color='red')
plt.title('Max Temperature Distribution')
plt.xlabel('Max Temperature (°C)')
plt.ylabel('Frequency')
plt.subplot(1, 3, 3)
sns.histplot(df_tmin['tmin_c'], bins=50, kde=True, color='green')
plt.title('Min Temperature Distribution')
plt.xlabel('Min Temperature (°C)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# create a fresh figure/axes to ensure plots render (don't rely on previously created axes)
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# (re)merge and drop NaNs to be safe
merge_tmin = pd.merge(df_prec[['cell_id','year','month','prec_mm']],
                      df_tmin[['cell_id','year','month','tmin_c']],
                      on=['cell_id','year','month']).dropna(subset=['prec_mm','tmin_c'])
merge_tmax = pd.merge(df_prec[['cell_id','year','month','prec_mm']],
                      df_tmax[['cell_id','year','month','tmax_c']],
                      on=['cell_id','year','month']).dropna(subset=['prec_mm','tmax_c'])

# compute Pearson correlations
r_tmin, p_tmin = pearsonr(merge_tmin['prec_mm'], merge_tmin['tmin_c'])
r_tmax, p_tmax = pearsonr(merge_tmax['prec_mm'], merge_tmax['tmax_c'])

# scatter prec vs tmin
axes[0].scatter(merge_tmin['prec_mm'], merge_tmin['tmin_c'], s=6, alpha=0.3, color='blue', marker='o')
axes[0].set_title(f'Precipitation vs Tmin (r={r_tmin:.3f}, p={p_tmin:.2e})')
axes[0].set_xlabel('Precipitation (mm)')
axes[0].set_ylabel('Tmin (°C)')
m, b = np.polyfit(merge_tmin['prec_mm'], merge_tmin['tmin_c'], 1)
xs = np.linspace(merge_tmin['prec_mm'].min(), merge_tmin['prec_mm'].max(), 100)
axes[0].plot(xs, m*xs + b, color='black', lw=1)

# scatter prec vs tmax
axes[1].scatter(merge_tmax['prec_mm'], merge_tmax['tmax_c'], s=6, alpha=0.3, color='red', marker='o')
axes[1].set_title(f'Precipitation vs Tmax (r={r_tmax:.3f}, p={p_tmax:.2e})')
axes[1].set_xlabel('Precipitation (mm)')
axes[1].set_ylabel('Tmax (°C)')
m2, b2 = np.polyfit(merge_tmax['prec_mm'], merge_tmax['tmax_c'], 1)
xs2 = np.linspace(merge_tmax['prec_mm'].min(), merge_tmax['prec_mm'].max(), 100)
axes[1].plot(xs2, m2*xs2 + b2, color='black', lw=1)


plt.tight_layout()
plt.show()

print(f'Prec vs Tmin: r = {r_tmin:.4f}, p = {p_tmin:.4e}')
print(f'Prec vs Tmax: r = {r_tmax:.4f}, p = {p_tmax:.4e}')


In [ ]:
plt.figure(figsize=(8,6))
data = [df_tmax['tmax_c'].dropna(), df_tmin['tmin_c'].dropna()]
plt.boxplot(data, notch=True, patch_artist=True,
            boxprops=dict(facecolor='lightgray', color='black'),
            medianprops=dict(color='red'),
            whiskerprops=dict(color='black'),
            capprops=dict(color='black'),
            flierprops=dict(marker='.', markerfacecolor='gray', markersize=3, alpha=0.5))
plt.xticks([1, 2], ['Tmax (°C)', 'Tmin (°C)'])
plt.title('Boxplots: Tmax and Tmin')
plt.ylabel('Temperature (°C)')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# loading soil dataset
df_soil = pd.read_csv('../../ExtractedDatasets/SoilFeatures/hwsd2_grid01_D1.csv')

In [ ]:
df_soil.shape

In [ ]:
#df_soil memory usage
print("Soil Data Memory Usage:", df_soil.memory_usage(deep=True).sum() / (1024 ** 2), "MB")

In [ ]:
df_soil.describe()

In [ ]:
#pourcentage of missing values in soil dataset
print(missing_percentage(df_soil),"%")

In [ ]:
# extracting soil empty rows
df_missing_soil = df_soil[df_soil.isnull().any(axis=1)]

print("Number of rows with missing soil data:", df_missing_soil.shape[0])

In [ ]:
# percentage of missing values in df_missing_soil for each column
print(df_missing_soil.isnull().mean() * 100)

In [ ]:
# check in df_missing_soil if  TEXTURE_USDA is missing  REF_BULK is not vise versa

print("Rows where both TEXTURE_USDA and REF_BULK are missing:")
# rows where both TEXTURE_USDA and REF_BULK are NOT missing (negative form: NOT)
print(df_missing_soil[(~(df_missing_soil['TEXTURE_USDA'].notnull()) & (df_missing_soil['REF_BULK'].notnull())) | (~(df_missing_soil['REF_BULK'].notnull()) & (df_missing_soil['TEXTURE_USDA'].notnull()))])


In [ ]:
# Boxplots: REF_BULK (subplot 1) and TEXTURE_USDA (subplot 2)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- REF_BULK ---
if 'REF_BULK' in df_soil.columns:
    ref_bulk = pd.to_numeric(df_soil['REF_BULK'], errors='coerce').dropna()
    if len(ref_bulk) > 0:
        axes[0].boxplot(ref_bulk, notch=True, patch_artist=True,
                        boxprops=dict(facecolor='lightgray', color='black'),
                        medianprops=dict(color='red'),
                        whiskerprops=dict(color='black'),
                        capprops=dict(color='black'),
                        flierprops=dict(marker='.', markerfacecolor='gray', markersize=3, alpha=0.5))
        axes[0].set_title('REF_BULK')
        axes[0].set_ylabel('REF_BULK')
    else:
        axes[0].text(0.5, 0.5, 'REF_BULK has no numeric values after coercion', ha='center', va='center')
        axes[0].set_axis_off()
else:
    axes[0].text(0.5, 0.5, 'Column REF_BULK not found', ha='center', va='center')
    axes[0].set_axis_off()

# --- TEXTURE_USDA ---
if 'TEXTURE_USDA' in df_soil.columns:
    # Try to coerce to numeric if it's encoded as strings; otherwise this will drop non-numeric
    texture_vals = pd.to_numeric(df_soil['TEXTURE_USDA'], errors='coerce').dropna()
    if len(texture_vals) > 0:
        axes[1].boxplot(texture_vals, notch=True, patch_artist=True,
                        boxprops=dict(facecolor='lightgray', color='black'),
                        medianprops=dict(color='red'),
                        whiskerprops=dict(color='black'),
                        capprops=dict(color='black'),
                        flierprops=dict(marker='.', markerfacecolor='gray', markersize=3, alpha=0.5))
        axes[1].set_title('TEXTURE_USDA')
        axes[1].set_ylabel('TEXTURE_USDA')
    else:
        axes[1].text(0.5, 0.5, 'TEXTURE_USDA not numeric or empty after coercion', ha='center', va='center')
        axes[1].set_axis_off()
else:
    axes[1].text(0.5, 0.5, 'Column TEXTURE_USDA not found', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Quantile plots (percentile vs value) for REF_BULK and TEXTURE_USDA
def _quantile_plot(series: pd.Series, ax, title: str):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.size < 2:
        ax.text(0.5, 0.5, f"{title}: not enough numeric data to plot quantiles", ha='center', va='center')
        ax.set_axis_off()
        return
    q = np.linspace(0, 1, 101)  # 0%, 1%, ..., 100%
    vals = np.quantile(s, q)
    # plot empirical quantile curve
    ax.plot(q * 100.0, vals, color='tab:blue', lw=2)
    ax.set_title(title)
    ax.set_xlabel('Percentile (%)')
    ax.set_ylabel('Value')
    ax.grid(alpha=0.3)
    
    # Mark quartiles Q1 (25%), Q2/median (50%), Q3 (75%), Q4 (100%)
    quartiles = [0.25, 0.50, 0.75, 1.00]
    q_labels = ['Q1', 'Q2', 'Q3', 'Q4']
    y_quart = np.quantile(s, quartiles)
    # subtle background bands for each quartile range
    bands = [(0, 25, 'Q1'), (25, 50, 'Q2'), (50, 75, 'Q3'), (75, 100, 'Q4')]
    band_colors = ['#fcebd6', '#e7f3fb', '#eaf7e6', '#fbe6ee']
    for (x0, x1, _lab), col in zip(bands, band_colors):
        ax.axvspan(x0, x1, color=col, alpha=0.25, zorder=0)
    # vertical lines at quartile boundaries
    for x in [25, 50, 75, 100]:
        ax.axvline(x, color='gray', lw=0.8, ls='--', alpha=0.6)
    # scatter markers and labels at quartile points
    ax.scatter([p * 100 for p in quartiles], y_quart, color='crimson', s=25, zorder=3)
    for p, y, lab in zip(quartiles, y_quart, q_labels):
        ax.annotate(lab, xy=(p * 100, y), xytext=(p * 100 + 1.5, y),
                    textcoords='data', fontsize=9, color='crimson', va='center')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# REF_BULK quantile plot
if 'REF_BULK' in df_soil.columns:
    _quantile_plot(df_soil['REF_BULK'], axes[0], 'REF_BULK quantiles')
else:
    axes[0].text(0.5, 0.5, 'Column REF_BULK not found', ha='center', va='center')
    axes[0].set_axis_off()

# TEXTURE_USDA quantile plot
if 'TEXTURE_USDA' in df_soil.columns:
    _quantile_plot(df_soil['TEXTURE_USDA'], axes[1], 'TEXTURE_USDA quantiles')
else:
    axes[1].text(0.5, 0.5, 'Column TEXTURE_USDA not found', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()